# الفصل 46: خط التقييم المستمر

> التقييم ليس حدثًا لمرة واحدة قبل الإطلاق — في الإنتاج الحقيقي، النماذج والبيانات والسلوكيات تتغيّر باستمرار (Model Drift، تحديثات، مستخدمون جدد).

- **الجزء:** 9 — التقييم الشامل
- **الفصل:** 46
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 45: التقييم النوعي وتحليل الأخطاء
- **الوسوم:** Continuous Evaluation، Drift Monitoring، CI/CD، Regression

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch46-continuous-evaluation.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> التقييم ليس حدثًا لمرة واحدة قبل الإطلاق — في الإنتاج الحقيقي، النماذج والبيانات والسلوكيات تتغيّر باستمرار (Model Drift، تحديثات، مستخدمون جدد).

> **بيانات الفصل**
>
> - **الجزء:** 9 — التقييم الشامل
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 45: التقييم النوعي وتحليل الأخطاء


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم معمارية خط التقييم المستمر
- فهم Continuous Evaluation
- فهم Drift Monitoring


## 1. لماذا يهمّك هذا الموضوع؟

التقييم ليس حدثًا لمرة واحدة قبل الإطلاق — في الإنتاج الحقيقي، النماذج والبيانات والسلوكيات تتغيّر باستمرار (Model Drift، تحديثات، مستخدمون جدد). هذا الفصل يعلّمك بناء نظام تقييم **آلي ومستمر** يُنذرك فور حدوث تراجع (regression).


## 2. الخريطة الذهنية


```text
خط التقييم المستمر
│
├── قبل الدمج (CI)
│   ├── Smoke tests (< 1 دقيقة)
│   ├── Golden Set ثابت
│   ├── Regression suite
│   └── Gate: لا دمج إذا تدهور > عتبة
│
├── قبل النشر (CD)
│   ├── Offline eval شامل
│   ├── Canary / Shadow
│   └── مقارنة بالنموذج الحالي
│
├── بعد النشر
│   ├── Online metrics (TTFT, error rate)
│   ├── Quality sampling
│   ├── Drift detection
│   └── Feedback loop
│
└── البنية
    ├── Versioned datasets
    ├── Experiment tracking
    └── Dashboards + alerts
```


## 3. الشرح المتدرج

### 🟢 أ) معمارية خط التقييم المستمر


```text
[نموذج جديد/تحديث]
        │
        ▼
[1. اختبارات الوحدة السريعة (Smoke Tests)] ──[فشل]──> رفض فوري
        │ [نجاح]
        ▼
[2. Benchmark Suite الكاملة (ليلاً/دوريًا)]
        │
        ▼
[3. LLM-as-a-Judge على Golden Set]
        │
        ▼
[4. مقارنة مع النموذج الحالي في الإنتاج (A/B قبل الإطلاق)]
        │
        ▼
[5. تقرير آلي + تنبيه عند أي تراجع (Regression Alert)]
        │
        ▼
[موافقة بشرية نهائية] ──> نشر تدريجي (Canary Deployment)
```


### 🟡 ب) الكود الكامل: نظام تقييم مستمر متكامل


In [ ]:
import json
import datetime
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class EvaluationSuite:
    """تعريف خط تقييم كامل قابل لإعادة الاستخدام عبر إصدارات النموذج"""
    smoke_tests: list = field(default_factory=list)
    benchmarks: dict = field(default_factory=dict)       # {name: eval_function}
    golden_set: list = field(default_factory=list)
    regression_thresholds: dict = field(default_factory=dict)  # {metric: min_acceptable_delta}

class ContinuousEvaluationPipeline:
    def __init__(self, suite: EvaluationSuite, baseline_model, candidate_model, judge_client):
        self.suite = suite
        self.baseline_model = baseline_model
        self.candidate_model = candidate_model
        self.judge_client = judge_client
        self.history = []

    def run_smoke_tests(self) -> bool:
        """اختبارات سريعة جدًا (ثوانٍ) — يجب أن تمر 100% وإلا رفض فوري"""
        for test in self.suite.smoke_tests:
            response = self.candidate_model.generate(test["prompt"])
            if not test["assertion_fn"](response):
                print(f"❌ فشل اختبار الدخان: {test['name']}")
                return False
        print("✅ كل اختبارات الدخان نجحت")
        return True

    def run_full_benchmarks(self) -> dict:
        results = {}
        for name, eval_fn in self.suite.benchmarks.items():
            score = eval_fn(self.candidate_model)
            results[name] = score
            print(f"📊 {name}: {score:.4f}")
        return results

    def run_golden_set_judge_comparison(self) -> dict:
        """مقارنة النموذج الجديد بالحالي على مجموعة ذهبية ثابتة"""
        wins, losses, ties = 0, 0, 0
        for example in self.suite.golden_set:
            response_baseline = self.baseline_model.generate(example["prompt"])
            response_candidate = self.candidate_model.generate(example["prompt"])

            verdict = pairwise_judge_with_position_swap(
                self.judge_client, example["prompt"], response_baseline, response_candidate
            )
            if verdict == "ب":   # النموذج المرشح أفضل
                wins += 1
            elif verdict == "أ":
                losses += 1
            else:
                ties += 1

        total = wins + losses + ties
        return {
            "win_rate": wins / total, "loss_rate": losses / total, "tie_rate": ties / total,
            "net_improvement": (wins - losses) / total
        }

    def check_regressions(self, benchmark_results: dict, baseline_results: dict) -> list:
        """كشف أي تراجع يتجاوز العتبة المسموحة"""
        regressions = []
        for metric, score in benchmark_results.items():
            baseline_score = baseline_results.get(metric, 0)
            delta = score - baseline_score
            threshold = self.suite.regression_thresholds.get(metric, -0.02)  # افتراضي: -2%
            if delta < threshold:
                regressions.append({
                    "metric": metric, "baseline": baseline_score,
                    "candidate": score, "delta": delta
                })
        return regressions

    def generate_report(self) -> dict:
        """تشغيل الخط الكامل وإنتاج تقرير نهائي"""
        report = {"timestamp": datetime.datetime.now().isoformat()}

        if not self.run_smoke_tests():
            report["status"] = "REJECTED_SMOKE_TEST_FAILURE"
            return report

        report["benchmark_results"] = self.run_full_benchmarks()
        report["judge_comparison"] = self.run_golden_set_judge_comparison()

        # افتراض وجود نتائج مرجعية محفوظة من قبل
        baseline_results = self._load_baseline_results()
        regressions = self.check_regressions(report["benchmark_results"], baseline_results)

        if regressions:
            report["status"] = "REGRESSION_DETECTED"
            report["regressions"] = regressions
        elif report["judge_comparison"]["net_improvement"] < -0.05:
            report["status"] = "QUALITY_DEGRADATION_DETECTED"
        else:
            report["status"] = "APPROVED_FOR_CANARY_DEPLOYMENT"

        self.history.append(report)
        return report

    def _load_baseline_results(self):
        # في التطبيق الفعلي: تحميل من قاعدة بيانات/ملف تتبّع تجارب (MLflow, W&B)
        return {}


### 🔴 ج) طبقة التعمّق: مراقبة الانجراف في الإنتاج (Production Drift Monitoring)


In [ ]:
class ProductionDriftMonitor:
    """
    مراقبة سلوك النموذج في الإنتاج الحقيقي (وليس فقط قبل النشر)
    لكشف تدهور تدريجي غير متوقع (Model/Data Drift)
    """
    def __init__(self, window_size=1000, alert_threshold=0.1):
        self.window_size = window_size
        self.alert_threshold = alert_threshold
        self.recent_scores = []
        self.baseline_mean = None

    def log_interaction_score(self, implicit_feedback_score: float):
        """
        implicit_feedback_score: إشارة ضمنية لجودة التفاعل
        (مثل: هل أعاد المستخدم صياغة السؤال؟ هل ضغط "إعجاب"؟ طول المحادثة؟)
        """
        self.recent_scores.append(implicit_feedback_score)
        if len(self.recent_scores) > self.window_size:
            self.recent_scores.pop(0)

        if self.baseline_mean is None and len(self.recent_scores) == self.window_size:
            self.baseline_mean = sum(self.recent_scores) / len(self.recent_scores)

    def check_drift(self) -> dict:
        if self.baseline_mean is None or len(self.recent_scores) < self.window_size:
            return {"status": "insufficient_data"}

        current_mean = sum(self.recent_scores) / len(self.recent_scores)
        relative_change = (current_mean - self.baseline_mean) / abs(self.baseline_mean)

        if abs(relative_change) > self.alert_threshold:
            return {
                "status": "DRIFT_DETECTED",
                "baseline": self.baseline_mean,
                "current": current_mean,
                "relative_change": relative_change
            }
        return {"status": "STABLE"}


> 💡 **نصيحة عملية**: في الإنتاج، لا تنتظر شكاوى المستخدمين لاكتشاف التدهور. اجمع **إشارات ضمنية** (implicit signals) مثل: معدل إعادة صياغة السؤال، معدل التخلي عن المحادثة، أزرار الإعجاب/عدم الإعجاب، طول وقت القراءة — وراقبها كمؤشرات رائدة (leading indicators) قبل أي انخفاض في مقاييس العمل الصريحة.


## 4. مثال محلول خطوة بخطوة

**مثال بوابة CI:**


```yaml
evaluation_gate:
  smoke_tests:
    max_duration_seconds: 60
    must_pass: 100%
  golden_set:
    size: 120
    max_regression: 2.0%   # مطلق
  safety:
    max_jailbreak_success: 1.0%
    max_false_refusal: 5.0%
  latency:
    max_p99_ms: 2500
```


**السيناريو:** النموذج الجديد يحقق 86.5% على Golden Set مقابل 88.0% للنموذج الحالي → تراجع 1.5 نقطة، أقل من 2% → يمر. لكنه يفشل في safety: 3% jailbreak → يُحجب.

**الدرس:** البوابة ليست رقمًا واحدًا؛ اجمع الجودة والأمان والزمن. وحدّد العتبات مسبقًا، لا بعد رؤية النتائج.


## 5. نصائح وحيل

> 💡 **اجعل Smoke tests سريعة**: أقل من دقيقة، 20-30 حالة تغطي المسارات الحرجة.

> 💡 **جمّد Golden Set**: لا تعدّله كل يوم؛ راجعه بشريًا كل ربع سنة.

> 💡 **استخدم Canary**: 1-5% من الزيارات للنموذج الجديد قبل التوسع.

> 💡 **اربط التنبيهات بالجودة لا بالأداء فقط**: TTFT ممتاز مع إجابات سيئة = فشل.

> 💡 **وثّق كل إصدار**: النموذج، الـprompt، البيانات، المقاييس، والقرار.


## 6. أخطاء شائعة ومصائد

> ⚠️ **تقييم لمرة واحدة قبل الإطلاق**: النماذج والبيانات تتغير. بدون CI، التراجع يصل للمستخدمين.

> ⚠️ **Golden Set صغير/متحيز**: 10 حالات لا تكفي. استهدف 100-200 حالة متنوعة.

> ⚠️ **الاعتماد على مقاييس تقنية فقط**: latency وerrors لا تكشف تدهور الجودة.

> ⚠️ **عتبات عشوائية**: حدّدها من احتياج العمل، لا من نتيجة تجربة واحدة.

> ⚠️ **إهمال التكلفة**: تقييم كل commit بنموذج قاضٍ مكلف. استخدم عينات ورتب تقييم.


## 7. جدول مقارنة

| المرحلة | نوع التقييم | التكرار |
|---|---|---|
| أثناء التطوير | Smoke tests + مقاييس سريعة | كل commit/تجربة |
| قبل الدمج (Merge) | Benchmark suite كاملة | يوميًا/لكل PR رئيسي |
| قبل النشر | LLM-as-a-Judge + Golden Set + Red Teaming | قبل كل إصدار |
| بعد النشر (Canary) | A/B Testing + مراقبة فورية | مستمر خلال فترة الـ Canary |
| في الإنتاج المستقر | Drift Monitoring + عيّنات دورية للمراجعة البشرية | مستمر + مراجعة أسبوعية |


## 8. ملخص أفضل الممارسات

- [ ] اختبارات دخان تعمل في أقل من دقيقة وتمنع أي انحدار كارثي واضح
- [ ] مجموعة Golden Set ثابتة ومُراجَعة بشريًا بعناية (50-200 مثال عالي الجودة)
- [ ] نظام تنبيه آلي عند أي تراجع يتجاوز عتبة محدَّدة مسبقًا
- [ ] نشر تدريجي (Canary/Blue-Green) وليس دفعة واحدة لكل التدفق
- [ ] مراقبة انجراف مستمرة بعد النشر عبر إشارات ضمنية وصريحة معًا
- [ ] حلقة تغذية راجعة (Feedback Loop) تُعيد أمثلة الفشل الحقيقية لتحسين مجموعات التقييم المستقبلية

---


## 9. أسئلة مقابلات

**س1: صمّم استراتيجية نشر آمنة لتحديث نموذج في الإنتاج.**

**الإجابة النموذجية:** (1) Offline eval كامل: smoke، Golden Set، safety، latency. (2) رفض أي تراجع يتجاوز عتبات محددة. (3) Canary 1-5% مع مراقبة الجودة والأداء. (4) Shadow traffic لمقارنة الاستجابات دون تأثير. (5) توسع تدريجي مع rollback سريع. (6) مراقبة ما بعد النشر وتغذية راجعة.

**س2: كيف تكشف الانجراف (Drift) في الإنتاج؟**

**الإجابة النموذجية:** راقب توزيع المدخلات (مواضيع، لغة، طول)، توزيع المخرجات (رفض، طول، أنماط)، والمقاييس الضمنية (إعادة صياغة، تخلٍّ، تقييمات). قارنها بbaseline واحسب انحرافات إحصائية. أضف عينات دورية بشرية.

**س3: ما الفرق بين Smoke tests وRegression suite؟**

**الإجابة النموذجية:** Smoke سريعة جدًا (<1 دقيقة) تتحقق من أن النظام يعمل ولا يوجد انحدار كارثي. Regression أوسع وأبطأ، تغطي حالات حرجة وتقيس فروقًا دقيقة. الأولى بوابة سريعة، الثانية بوابة قبل النشر.


## 10. تمرين عملي

1. صمّم `EvaluationSuite` لروبوت دعم عملاء عربي: 5 smoke، 3 benchmarks، 20 Golden Set.
2. حدّد عتبات القبول للجودة والأمان والزمن.
3. اكتب سيناريو Canary مع نسب وrollback.
4. صمّم لوحة مراقبة بعد النشر (TTFT، p99، refusal rate، drift).
5. نفّذ نموذجًا أوليًا بـPython يقارن نتائج إصدارين ويطبع قرار البوابة.


## 11. مراجع للتعمّق

- Breck, E., et al. (2017). *The ML Test Score: A Rubric for ML Production Readiness*.
- Sculley, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS.
- Google SRE Team. *Site Reliability Engineering* (O'Reilly).
- OpenAI Evals — https://github.com/openai/evals


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التقييم الشامل / الفصل 46: خط التقييم المستمر.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
